# 5.19 Sampling (rejection, importance)

**Lesson tagline.** Sampling estimates integrals by drawing from something easy and correcting toward the target.

Sampling is the Monte Carlo foundation for particle filters, MCMC, and HMC. Here we estimate expectations under difficult target distributions by using proposals, weights, and rejection envelopes while watching support and variance. Save a copy to Drive to edit.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

SEED = 519
rng = np.random.default_rng(SEED)

def normalize(values):
    arr = np.asarray(values, dtype=float)
    total = arr.sum()
    if total <= 0:
        raise ValueError("probabilities must have positive mass")
    return arr / total

def effective_sample_size(weights):
    w = np.asarray(weights, dtype=float)
    total = w.sum()
    if total == 0:
        return 0.0
    normalized = w / total
    return 1.0 / np.sum(normalized ** 2)

def discrete_draw(probs, size, random_state):
    probs = normalize(probs)
    return random_state.choice(len(probs), size=size, p=probs)

def sample_from_proposal(rung, n, random_state):
    if rung["kind"] == "discrete":
        return discrete_draw(rung["proposal"], n, random_state)
    if rung["kind"] == "mixture_1d":
        flags = random_state.random(n) < rung["proposal_mix"]
        samples = np.empty(n)
        samples[flags] = random_state.normal(rung["proposal_means"][0], rung["proposal_scales"][0], flags.sum())
        samples[~flags] = random_state.normal(rung["proposal_means"][1], rung["proposal_scales"][1], (~flags).sum())
        return samples
    if rung["kind"] == "gaussian":
        return random_state.multivariate_normal(rung["proposal_mean"], rung["proposal_cov"], size=n)
    raise ValueError(rung["kind"])

def gaussian_pdf(x, mean, cov):
    x = np.atleast_2d(x)
    mean = np.asarray(mean)
    cov = np.asarray(cov)
    inv = np.linalg.inv(cov)
    diff = x - mean
    exponent = -0.5 * np.sum(diff @ inv * diff, axis=1)
    norm = np.sqrt(((2 * np.pi) ** len(mean)) * np.linalg.det(cov))
    return np.exp(exponent) / norm

def mixture_pdf_1d(x, means, scales, weights):
    x = np.asarray(x)
    density = np.zeros_like(x, dtype=float)
    for mean, scale, weight in zip(means, scales, weights):
        z = (x - mean) / scale
        density += weight * np.exp(-0.5 * z ** 2) / (np.sqrt(2 * np.pi) * scale)
    return density

def build_sampling_ladder():
    ladder = []
    ladder.append({"name": "D1 2-state exact", "kind": "discrete", "target": np.array([0.35, 0.65]), "proposal": np.array([0.5, 0.5]), "h": np.array([0.0, 1.0]), "truth": 0.65})
    ladder.append({"name": "D2 4-state lesson target", "kind": "discrete", "target": np.array([0.1, 0.2, 0.4, 0.3]), "proposal": np.ones(4) / 4, "h": np.arange(4), "truth": 1.9})
    ladder.append({"name": "D3 bimodal target", "kind": "mixture_1d", "target_means": [-2.0, 2.0], "target_scales": [0.45, 0.65], "target_weights": [0.45, 0.55], "proposal_means": [0.0, 0.0], "proposal_scales": [1.0, 2.0], "proposal_mix": 0.85, "truth": 0.2})
    ladder.append({"name": "D4 correlated 2-D", "kind": "gaussian", "target_mean": np.array([0.5, -0.5]), "target_cov": np.array([[1.0, 0.85], [0.85, 1.4]]), "proposal_mean": np.array([0.0, 0.0]), "proposal_cov": np.eye(2) * 2.5, "truth": 0.5})
    ladder.append({"name": "D5 high-dim near-missing", "kind": "gaussian", "target_mean": np.array([0.0, 1.0, -1.0, 0.5, -0.5]), "target_cov": np.diag([0.15, 0.4, 1.0, 2.0, 4.0]), "proposal_mean": np.zeros(5), "proposal_cov": np.diag([0.04, 0.2, 0.8, 1.0, 1.0]), "truth": 0.0})
    return ladder

def target_density(rung, x):
    if rung["kind"] == "discrete":
        return rung["target"][np.asarray(x, dtype=int)]
    if rung["kind"] == "mixture_1d":
        return mixture_pdf_1d(x, rung["target_means"], rung["target_scales"], rung["target_weights"])
    if rung["kind"] == "gaussian":
        return gaussian_pdf(x, rung["target_mean"], rung["target_cov"])
    raise ValueError(rung["kind"])

def proposal_density(rung, x):
    if rung["kind"] == "discrete":
        return rung["proposal"][np.asarray(x, dtype=int)]
    if rung["kind"] == "mixture_1d":
        return mixture_pdf_1d(x, rung["proposal_means"], rung["proposal_scales"], [rung["proposal_mix"], 1 - rung["proposal_mix"]])
    if rung["kind"] == "gaussian":
        return gaussian_pdf(x, rung["proposal_mean"], rung["proposal_cov"])
    raise ValueError(rung["kind"])

def summary_stat(rung, samples):
    if rung["kind"] == "discrete":
        return rung["h"][samples]
    if rung["kind"] == "mixture_1d":
        return samples
    return samples[:, 0]

## The concept, built once (D1)
Importance sampling rewrites an expectation under a hard target $p$ as one under an easier proposal $q$:
$$\mathbb E_p[h(X)] = \mathbb E_q\left[h(X)\frac{p(X)}{q(X)}\right].$$
We first compute the weights and the normalized estimate, because every later rung uses the same correction.

In [ ]:
def importance_estimate(target, proposal, h):
    target = normalize(target)
    proposal = normalize(proposal)
    if np.any((proposal == 0) & (target > 0)):
        raise ValueError("proposal has missing target support")
    weights = target / proposal
    estimate = float(np.sum(proposal * weights * h))
    ess = effective_sample_size(weights)
    return weights, estimate, ess

lesson_target = np.array([0.1, 0.2, 0.4, 0.3])
lesson_proposal = np.ones(4) / 4
lesson_h = np.arange(4)
ratios, estimate, ess = importance_estimate(lesson_target, lesson_proposal, lesson_h)
assert np.allclose(ratios, [0.4, 0.8, 1.6, 1.2])
assert abs(estimate - 1.9) < 1e-12
assert ess < 4.0
print("ratios", ratios)
print("estimate", estimate)
print("ESS", ess)

A rejection sampler needs an envelope $M q(x)$ with $p(x) \le Mq(x)$ everywhere. For normalized discrete distributions, a tight envelope has acceptance probability $1/M$; the lesson uses $M=1.6$, so the expected acceptance is $0.625$.

In [ ]:
def rejection_sample(target, proposal, n, random_state):
    target = normalize(target)
    proposal = normalize(proposal)
    ratios = target / proposal
    envelope = float(np.max(ratios))
    accepted = []
    attempts = 0
    while len(accepted) < n:
        candidate = random_state.choice(len(proposal), p=proposal)
        accept_prob = target[candidate] / (envelope * proposal[candidate])
        attempts += 1
        if random_state.random() < accept_prob:
            accepted.append(candidate)
    return np.array(accepted), envelope, n / attempts

samples, envelope, observed_acceptance = rejection_sample(lesson_target, lesson_proposal, 200, rng)
expected_acceptance = 1.0 / envelope
assert abs(envelope - 1.6) < 1e-12
assert abs(expected_acceptance - 0.625) < 1e-12
print("M", envelope)
print("expected acceptance", expected_acceptance)
print("observed acceptance", observed_acceptance)

## The dataset ladder
Family F10 is built inline as synthetic target distributions of rising complexity: 2-state, 4-state, bimodal, correlated 2-D, then high-dimensional and ill-conditioned with near-missing proposal support.

In [ ]:
ladder = build_sampling_ladder()
for i, rung in enumerate(ladder, start=1):
    preview = sample_from_proposal(rung, 6, rng)
    print(i, rung["name"], "kind=", rung["kind"])
    print("sample", np.asarray(preview))

## Run the SAME method across D1-D5
Collect the plan metric on every rung from D1–D5 with the same algorithmic implementation, then print a compact per-rung table.

In [ ]:
sizes = [50, 100, 250, 500, 1000]
rows = []
traces = {}
for rung in ladder:
    samples = sample_from_proposal(rung, sizes[-1], rng)
    target = target_density(rung, samples)
    proposal = proposal_density(rung, samples)
    weights = target / np.maximum(proposal, 1e-300)
    stat = summary_stat(rung, samples)
    estimates = []
    errors = []
    for n in sizes:
        w = weights[:n]
        y = stat[:n]
        estimate_n = float(np.sum(w * y) / np.sum(w))
        estimates.append(estimate_n)
        errors.append(abs(estimate_n - rung["truth"]))
    ess = effective_sample_size(weights)
    rows.append((rung["name"], errors[-1], ess))
    traces[rung["name"]] = {"samples": samples, "weights": weights, "errors": errors, "estimates": estimates}
print("rung | abs expectation error | ESS")
for row in rows:
    print(f"{row[0]} | {row[1]:.4f} | {row[2]:.1f}")

## Results visualization
The closing figure has target/sample panels on top and the requested error-vs-iteration or error-vs-sample curve below.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, rung in zip(axes[0], ladder):
    info = traces[rung["name"]]
    samples = info["samples"]
    weights = info["weights"]
    if rung["kind"] == "discrete":
        states = np.arange(len(rung["target"]))
        weighted = np.bincount(samples, weights=weights, minlength=len(states))
        weighted = weighted / weighted.sum()
        ax.bar(states - 0.15, rung["target"], width=0.3, label="target")
        ax.bar(states + 0.15, weighted, width=0.3, label="weighted")
    else:
        values = summary_stat(rung, samples)
        ax.hist(values, bins=30, weights=weights / weights.sum(), alpha=0.7)
        ax.axvline(rung["truth"], color="black", linestyle="--")
    ax.set_title(rung["name"], fontsize=8)
for ax, rung in zip(axes[1], ladder):
    ax.plot(sizes, traces[rung["name"]]["errors"], marker="o")
    ax.set_xscale("log")
    ax.set_title("error vs samples", fontsize=8)
axes[0, 0].legend(fontsize=8)
fig.tight_layout()
plt.show()

## Pitfall on the hardest rung
The D5 pitfall is missing or near-missing support: if $q(x)=0$ where $p(x)>0$, weights cannot repair the estimate. Even near-missing tails create a tiny ESS. The fix is a heavier-tailed or mixture proposal that covers the target.

In [ ]:
d5 = ladder[-1]
bad = dict(d5)
bad["proposal_cov"] = np.diag([0.01, 0.05, 0.2, 0.2, 0.2])
good = dict(d5)
good["proposal_cov"] = d5["target_cov"] * 2.5
for label, rung in [("bad near-missing", bad), ("fixed heavy-tailed", good)]:
    samples = sample_from_proposal(rung, 1500, rng)
    weights = target_density(d5, samples) / np.maximum(proposal_density(rung, samples), 1e-300)
    estimate = np.sum(weights * samples[:, 0]) / np.sum(weights)
    error = abs(estimate - d5["truth"])
    print(label, "error=", round(float(error), 4), "ESS=", round(effective_sample_size(weights), 1))

## Evaluate it + Practice
- **Metric.** ESS and absolute expectation error on every rung.
- **No-skill baseline.** Use raw proposal samples without weights.
- **Cheap sanity check.** Weights should be nonnegative and normalized weighted masses should sum to 1.
- **Ablation.** Replace the mixture/heavy-tailed proposal with a narrow Gaussian and watch ESS collapse.
- **Failure signals.** Infinite weights, ESS near 1, or estimates that jump when one sample changes.

### Practice prompts
1. Swap the D3 proposal mixture weights and measure ESS.
2. Estimate a nonlinear expectation such as $E[X^2]$ on D3.
3. Add a support check that raises before any weights are formed.

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Your code here